# 5 · Evaluation

Stage 6 of the chain. The one place both architectures are scored, so that `4a` and `4b` are
comparable by construction rather than by convention.

**Position** `3_representations` → `4a` → `4b` → **`5_evaluation`**

| | |
|---|---|
| **reads** | `outputs/panel/panel_oof_predictions.csv` — `4a`, one row per (arm, drug, cell line) |
| | `outputs/mil/mil_oof_predictions.csv` — `4b`, same layout |
| | `outputs/panel/panel_ridge_baseline.csv` — the ridge control |
| **writes** | `outputs/panel/panel_metrics.csv` — four quantities per arm |

**Three sections, and they do not run at the same point.** §1 scores the arms and needs only the
out-of-fold predictions. §2 absorbs the external DrEval benchmark and §3 the diagnostics; both
consume retrained outputs and stay in `analysis/evaluation/` until they run. Running this notebook
top to bottom before those exist fails halfway, by design.

**The four quantities.** *Order* — within a drug, does the predicted ranking of cell lines match the
true one. *Top-of-order* — are the lines called most extreme actually the most extreme.
*Values* — how far off, against a per-drug constant. *Spread* — does the model use the real range or
collapse toward the mean.

**Why four and not one.** Every intervention this project has judged was judged on rank correlation
and MSE, and neither can see a change in *calibration*: widening the predictions without reordering
them leaves Spearman identical, and MSE is minimised by the shrunken predictor to begin with. So the
density weighting — whose effect is on spread, not order — was invisible to both metrics that were
asked about it.

---

# §1 · Score the arms

| | |
|---|---|
| **in** | the out-of-fold tables of `4a` and `4b`, keyed by `rep, model, alpha, loss, seed` |
| **out** | `panel_metrics.csv` — one row per arm, four quantities plus their drug counts |

Every row of the input is one (cell line × drug) pair, predicted by the fold that held that line out.
Nothing here trains anything.

**Why it runs at R4 rather than after.** The loss comparison cannot be judged without the calibration
slope below, so §1 is needed at the same point the arms are produced, not once everything else is
finished.

In [1]:
import os
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts').is_dir())
NB_DIR = ROOT / 'notebooks'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import numpy as np
import pandas as pd

# Named for the subdirectory it actually writes to, not for the outputs root (routed here by the
# code-quality session, 13.08.2026 -- the same flat-root trap as item #24). `OUT = NB_DIR/'outputs'`
# with every call site re-appending 'panel' is correct exactly until one call site forgets, and then
# it writes to the root and nothing complains. Nothing lands wrong today; the NEXT write is the one
# that would.
PANEL_OUT = NB_DIR / 'outputs' / 'panel'
OOF_CSV = PANEL_OUT / 'panel_oof_predictions.csv'
METRICS_CSV = PANEL_OUT / 'panel_metrics.csv'

#: Columns that together identify one training configuration. Settled 13.08.2026: 4a now stamps all
#: five. `alpha` is the density-weighting level (0.0/0.5/1.0) and is meaningful for model='mlp' rows
#: only; `model` separates mlp / ridge / mil so no column carries both a level and a name.
ARM_KEYS = ['rep', 'model', 'alpha', 'loss', 'seed']

#: Columns every row must carry regardless of how the arm is keyed. `fold` is required for the same
#: reason dreval_normalize requires it: a prediction that cannot be traced to the split that produced
#: it cannot have an out-of-fold baseline fitted against it.
BASE_COLS = ['drug', 'cell_line', 'fold', 'y_true', 'y_pred']


def load_oof_table(path=OOF_CSV, arm_keys=ARM_KEYS):
    """Read the out-of-fold predictions and refuse the table if scoring it would be misleading.

    Raises rather than warning, in every case. A warning printed above a table of plausible numbers
    is read once and then scrolled past; this notebook exists to decide which loss arm wins, and a
    decision taken on a quietly malformed table is the failure it is meant to prevent.
    """
    if not path.exists():
        raise FileNotFoundError(
            f'{path} does not exist. It is written by 4a_percell_training at R4 of the sweep. '
            f'There is deliberately no fallback to a cached copy: every committed out-of-fold file '
            f'predates the early-stopping fix and the panel rebuild.'
        )
    oof = pd.read_csv(path)

    missing = [c for c in [*BASE_COLS, *arm_keys] if c not in oof.columns]
    if missing:
        raise ValueError(f'{path.name} is missing {missing}. Present: {list(oof.columns)}.')

    nan_rows = oof[['y_true', 'y_pred']].isna().any(axis=1).sum()
    if nan_rows:
        raise ValueError(f'{nan_rows} rows have a null y_true or y_pred; every scored pair needs both.')

    # THE CHECK THAT MATTERS. A duplicate key means some dimension of the run is not stamped on the
    # rows, so distinct predictions are about to be averaged together without anything saying so.
    key = ['drug', 'cell_line', *arm_keys]
    dup = oof.duplicated(subset=key, keep=False)
    if dup.any():
        ex = oof[dup].sort_values(key).head(6)
        raise ValueError(
            f'{int(dup.sum())} rows share a (drug, cell_line, {", ".join(arm_keys)}) key, so more than '
            f'one prediction claims the same slot. Some dimension of the run is not stamped on the '
            f'rows -- seed and the loss arm are the expected culprits. Add it to ARM_KEYS *and* to the '
            f'columns 4a stamps; do not deduplicate. Averaging across seeds here would measure the '
            f'seed band as zero, and the decision rule reads its margin off that band.\n{ex}'
        )

    # One cell line belongs to exactly one fold within an arm, or the split was not grouped by line.
    per_arm_folds = oof.groupby([*arm_keys, 'cell_line'])['fold'].nunique()
    if (per_arm_folds > 1).any():
        bad = per_arm_folds[per_arm_folds > 1]
        raise ValueError(f'{len(bad)} cell lines are held out by more than one fold within an arm:\n{bad.head()}')

    # Every arm must cover the same pairs, or the arms are not being compared on the same data.
    #
    # ⛔ THIS ASSERTION MAY NOW BE WRONG ABOUT ITS OWN SUBJECT -- unverified, flagged 13.08.2026.
    # It was written when an arm was (rep, weighted) and every arm was an MLP. `model` now separates
    # mlp / ridge / mil, and the ridge baseline is fitted per drug on line-mean embeddings, so it may
    # legitimately cover a different set of (drug, cell_line) pairs than the MLP arms do. If it does,
    # this raises at R4 ON CORRECT DATA, which is worse than not checking: a spurious failure trains
    # people to bypass the check that catches the real one. Decide before R4 whether the comparison
    # is within-model (group by `model` first, require identical pairs inside each group) or across
    # models (require identical pairs everywhere, and make ridge conform). Not resolved here --
    # which arms are compared with which is an analysis decision.
    pair_sets = {arm: frozenset(map(tuple, g[['drug', 'cell_line']].to_numpy()))
                 for arm, g in oof.groupby(arm_keys, sort=False)}
    sizes = {a: len(s) for a, s in pair_sets.items()}
    if len(set(pair_sets.values())) > 1:
        raise ValueError(
            f'The arms do not cover identical (drug, cell_line) pairs, so any comparison between them '
            f'is partly a comparison of which pairs they were scored on. Pairs per arm: {sizes}.'
        )

    return oof


oof = load_oof_table()
print(f'{len(oof):,} rows from {OOF_CSV.name}')
print(f'  arms      : {oof.groupby(ARM_KEYS, sort=False).ngroups}  keyed by {ARM_KEYS}')
print(f'  drugs     : {oof.drug.nunique()}')
print(f'  cell lines: {oof.cell_line.nunique()}')
print(f'  folds     : {sorted(oof.fold.unique())}')
print(f'  pairs/arm : {len(oof) // max(oof.groupby(ARM_KEYS, sort=False).ngroups, 1):,}')


58,860 rows from panel_oof_predictions.csv
  arms      : 36  keyed by ['rep', 'model', 'alpha', 'loss', 'seed']
  drugs     : 11
  cell lines: 153
  folds     : [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
  pairs/arm : 1,635


## 1.2 · Spread — the calibration slope, with its intercept

| | |
|---|---|
| **in** | per drug: true and predicted line-level responses |
| **out** | `(slope, intercept)`; `nan` where undefined |

Regress **truth on prediction**. A perfectly calibrated model gives slope 1: one unit of predicted
response corresponds to one unit of true response. Slope below 1 means the predictions are spread
wider than their accuracy supports; above 1 means they are shrunk toward the mean.

**Why the pair and not a ratio of standard deviations.** The slope catches compression and the
intercept catches shift, and a model can be flat *and* shifted — a ratio of spreads sees neither
separately. Van Calster, Nieboer, Vickers, Van Calster & Steyerberg, *A calibration hierarchy for
risk models*, J Clin Epidemiol 74 (2016) 167–176.

Both are `nan` when the fit is not defined — fewer than three pairs, or a prediction vector with no
variance, which is the total-collapse case and must be visible rather than silently scored.

In [2]:
from scipy.stats import linregress


def calibration(y_true, y_pred, *, min_points=3):
    """Calibration slope and intercept from regressing TRUTH on PREDICTION.

    :returns: ``(slope, intercept)``. Both ``nan`` when the fit is not defined -- fewer than
        ``min_points`` pairs, or a prediction vector with no variance (a model that emitted one
        constant for every cell line of this drug, which is the total-collapse case the slope exists
        to detect). ``nan`` is returned rather than 0 or inf deliberately: a collapsed arm must show
        up as "not measurable here" in the summary, not as a number that averages with real slopes.

    Van Calster et al., J Clin Epidemiol 74 (2016) 167-176 -- their weak-calibration level.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if y_true.shape != y_pred.shape:
        raise ValueError(f'shape mismatch: y_true {y_true.shape} vs y_pred {y_pred.shape}')
    if y_true.size < min_points or np.ptp(y_pred) == 0:
        return np.nan, np.nan
    fit = linregress(y_pred, y_true)      # x = prediction, y = truth -- the direction matters
    return float(fit.slope), float(fit.intercept)


## 1.3 · The decision rule

| | |
|---|---|
| **in** | `{arm: {quantity: [one value per seed]}}`, at least three seeds |
| **out** | verdict plus every margin and difference it used |

An arm wins on **order**; the other three act as non-inferiority guards. A challenger takes the
primary quantity and is worse on no guard by more than that guard's margin.

**Why the rule is written before the run.** The previous loss comparison failed because its rule was
never written down: it was judged on Spearman and MSE, both structurally blind to the effect the
weighting was supposed to have, and that only became visible afterwards. A rule fixed after the
numbers are visible is not a rule.

**Why the margin is per quantity.** ±0.04 is the seed band on Spearman, so it is the right bar for
*order* and transfers to nothing else: *values* is an error in the target's own units and the
calibration slope is centred on 1.0 on a different scale, so one number would mean three different
things and two of them would be unsourced.

**Why `SEED_BAND` refuses to default.** On three seeds a half-range and a standard deviation differ
by about a factor of two, and how the recorded ±0.04 was computed is not documented — so it cannot be
inherited. The rule raises until the band is stated.

In [3]:
#: The quantity that decides. Selin, 13.08.2026.
PRIMARY_QUANTITY = 'order'
GUARD_QUANTITIES = ('top_of_order', 'values', 'spread_slope')

#: What "better" means per quantity -- see the table above. 'toward_one' is the calibration slope.
DIRECTION = {'order': 'higher', 'top_of_order': 'higher',
             'values': 'lower', 'spread_slope': 'toward_one'}

#: Recorded prior estimate of order's seed band (docs/steps/03, docs/TODO.md). NOT a default: the
#: margins are measured from the run. Kept so a measured band wildly unlike it is noticed.
PRIOR_ORDER_BAND = 0.04

#: ⛔ UNSET ON PURPOSE. How a quantity's seed band is computed from its >=3 per-seed values.
#: Fill with a callable, e.g.  lambda v: 0.5 * (np.max(v) - np.min(v))   (half-range)
#:                       or   lambda v: float(np.std(v, ddof=1))          (sample sd)
#: These differ by ~2x on three seeds and that factor IS the decision bar, so it is a person's
#: choice. How the recorded 0.04 was computed is not written down anywhere, so it cannot be inherited.
SEED_BAND = None


def _badness(quantity, value):
    """Map a quantity to a number where LOWER IS ALWAYS WORSE-IS-BIGGER, so one comparison serves all."""
    how = DIRECTION[quantity]
    if how == 'higher':
        return -value
    if how == 'lower':
        return value
    if how == 'toward_one':
        return abs(value - 1.0)
    raise ValueError(f'unknown direction {how!r} for {quantity!r}')


def decide(per_seed, challenger, incumbent, *, seed_band=None):
    """Apply the rule: win on the primary, non-inferior on every guard.

    :param per_seed: {arm: {quantity: [one value per seed]}}. Every arm needs the same quantities
        and at least three seeds; anything less is not a comparison this rule can make.
    :param seed_band: overrides SEED_BAND. Pass a fixed float per quantity via a dict instead of a
        callable if Selin replaces a measured band with a chosen number.
    :returns: dict with the verdict and, importantly, every margin and difference it used -- so the
        decision can be read afterwards rather than recomputed to be believed.
    """
    band = seed_band if seed_band is not None else SEED_BAND
    if band is None:
        raise ValueError(
            'SEED_BAND is unset. It defines the margin every comparison is judged against, and on '
            'three seeds a half-range and an sd differ by about a factor of two. How the recorded '
            '+-0.04 was computed is not documented, so it cannot be inherited -- see the markdown '
            'above. Set SEED_BAND, or pass seed_band=, before any arm is declared a winner.'
        )

    quantities = (PRIMARY_QUANTITY, *GUARD_QUANTITIES)
    for arm in (challenger, incumbent):
        if arm not in per_seed:
            raise KeyError(f'arm {arm!r} not in the per-seed table')
        for q in quantities:
            n = len(per_seed[arm].get(q, ()))
            if n < 3:
                raise ValueError(
                    f'{arm!r} has {n} seed(s) for {q!r}; the rule needs >=3 because the margin is '
                    f'the spread across seeds. With fewer, the bar is undefined, not merely noisy.'
                )

    def margin(q):
        vals = [*per_seed[challenger][q], *per_seed[incumbent][q]]
        return float(band(vals)) if callable(band) else float(band[q])

    report, verdict = {}, True
    for q in quantities:
        c = _badness(q, float(np.mean(per_seed[challenger][q])))
        i = _badness(q, float(np.mean(per_seed[incumbent][q])))
        m = margin(q)
        improvement = i - c                      # positive = challenger is better
        passed = improvement > m if q == PRIMARY_QUANTITY else improvement > -m
        report[q] = {'challenger': c, 'incumbent': i, 'improvement': improvement,
                     'margin': m, 'role': 'primary' if q == PRIMARY_QUANTITY else 'guard',
                     'passed': bool(passed)}
        verdict &= bool(passed)

    return {'challenger': challenger, 'incumbent': incumbent,
            'challenger_wins': bool(verdict), 'per_quantity': report}


## 1.4 · Order — within-drug Spearman

| | |
|---|---|
| **in** | per drug: true and predicted line-level responses |
| **out** | ρ, or `nan` when undefined |

Spearman within each drug across the held-out cell lines: does the predicted ranking of lines match
the true one?

**Why within drug and not pooled.** Pooling across drugs would let a model score well by learning
only that some compounds are broadly more potent than others — a between-drug effect that says
nothing about which cell lines respond. The question is per compound, so the statistic is too.

**Why rank and not Pearson.** The target is a curve summary on a bounded scale with a long tail;
order is what a ranking of candidate lines needs, and it is invariant to any monotone rescaling of
the target — which matters when the target itself has been replaced twice.

In [4]:
from scipy.stats import spearmanr


def order(y_true, y_pred, *, min_points=3):
    """Spearman rank correlation between the predicted and true ordering of cell lines, for one drug.

    Applied to a drug's rows across ALL folds at once -- that is what "pooled across folds" means,
    and it is realised by the caller not grouping on ``fold`` rather than by any flag here.

    :returns: rho, or ``nan`` when it is not defined -- fewer than ``min_points`` lines, or a constant
        vector on either side. ``nan`` rather than 0.0 deliberately, matching :func:`calibration`: a
        drug that cannot be ranked must read as "not measurable here" and be excluded from the mean,
        not silently contribute a zero that drags the arm's score toward the middle.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if y_true.shape != y_pred.shape:
        raise ValueError(f'shape mismatch: y_true {y_true.shape} vs y_pred {y_pred.shape}')
    if y_true.size < min_points or np.ptp(y_true) == 0 or np.ptp(y_pred) == 0:
        return np.nan
    rho = spearmanr(y_pred, y_true).statistic
    return float(rho) if np.isfinite(rho) else np.nan


## 1.5 · Top-of-order — Precision@20

| | |
|---|---|
| **in** | per drug: true and predicted line-level responses; `k = 20` |
| **out** | the fraction of the true top-k the model also places in its top-k |

**Why a separate quantity from order.** Spearman weights the whole ranking equally, but the
clinically interesting lines are at one end. A model can hold a respectable ρ while misplacing every
extreme line, and that failure is invisible to the primary quantity.

**Why k = 20.** Roughly 13 % of the ~150 held-out lines per drug — large enough that the statistic is
not dominated by one or two lines, small enough to still be "the top". It is a chosen number and is
stated as one; the quantity is reported for its own sake rather than as a gate.

In [5]:
#: How many lines count as "the top". 20 of ~150 held-out lines per drug, ~13%. Not a display
#: setting: a smaller K is noisier seed-to-seed, and the seed spread IS the decision margin, so K
#: sets how hard the comparison is to pass. Selin, 13.08.2026.
TOP_K = 20


def top_of_order(y_true, y_pred, *, k=TOP_K):
    """Precision@K at the SENSITIVE end: of the k lines predicted most sensitive, how many truly are.

    Sensitive means LOW AUC, so both rankings are ASCENDING. There is no direction parameter on
    purpose -- reversing it would silently report resistance-finding as sensitivity-finding, and no
    output would look wrong. Changing the end means editing this function.

    :returns: a fraction in [0, 1], or ``nan`` when the question is not meaningful -- specifically
        when a drug has ``k`` lines or fewer, where both sets are the whole set and the score is a
        trivial 1.0. Returning that 1.0 would quietly reward thin coverage.

    ⚠️ Ties are broken by ``argsort`` order, which is arbitrary. With continuous AUC exact ties are
    rare, but a drug measured on a coarse grid could have several at the boundary, and then this
    number depends on row order rather than on the model.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if y_true.shape != y_pred.shape:
        raise ValueError(f'shape mismatch: y_true {y_true.shape} vs y_pred {y_pred.shape}')
    if y_true.size <= k:
        return np.nan
    true_top = set(np.argsort(y_true)[:k].tolist())     # ascending: lowest AUC = most sensitive
    pred_top = set(np.argsort(y_pred)[:k].tolist())
    return len(true_top & pred_top) / k


## 1.6 · From per-drug statistics to one number per arm

| | |
|---|---|
| **in** | `{drug: value}` from any of the four quantities |
| **out** | `(mean, drugs_used)` |

The mean over drugs where the statistic is defined, plus the list of those drugs.

**Why the list is returned.** Two arms averaged over different drug sets are not comparable, and the
difference is invisible in the means alone. A caller can check that it is comparing like with like
rather than assuming it.

**Why an unweighted mean.** Weighting by coverage would let the best-covered compounds dominate, and
coverage is a property of the screen rather than of the biology. Every panel drug clears 90 %
coverage by construction, so the weights would be nearly equal anyway — and stating that is cheaper
than hiding it in a weighted average.

In [6]:
def per_drug(frame, statistic):
    """Apply a per-drug statistic to one arm's rows, pooled across folds.

    Grouped by drug ONLY. The absence of ``fold`` in the grouping is the "pooled across folds"
    decision (Selin, 13.08.2026) -- each held-out line appears exactly once across the folds, so
    pooling uses each line once. Do not add ``fold`` here without changing that decision.

    :param statistic: ``f(y_true, y_pred) -> float``, e.g. :func:`order` or :func:`top_of_order`.
    :returns: ``{drug: value}``, values possibly ``nan`` where the statistic is undefined.
    """
    return {drug: statistic(g['y_true'].to_numpy(), g['y_pred'].to_numpy())
            for drug, g in frame.groupby('drug', sort=False)}


def across_drugs(values):
    """The unweighted mean over drugs -- every drug counts once, whatever its coverage.

    Unweighted was chosen for continuity with every number already recorded, knowing the cost: a
    thinly covered, noisier drug moves the mean as much as a well covered one (§1.4).

    ``nan`` drugs are EXCLUDED rather than counted as zero -- a drug that cannot be ranked is a
    property of the data, not a failure of the arm.

    :returns: ``(mean, drugs_used)``. The second element exists so a caller can tell whether two arms
        were averaged over the same panel; see the warning in the markdown above.
    """
    used = sorted(d for d, v in values.items() if v is not None and np.isfinite(v))
    if not used:
        return np.nan, []
    return float(np.mean([values[d] for d in used])), used


## 1.7 · Values — mean absolute error

| | |
|---|---|
| **in** | per drug: true and predicted line-level responses; the per-drug constant fitted out of fold |
| **out** | MAE, and the MAE of the constant it is read against |

**Why absolute and not squared.** It is in the target's own units and directly readable — an MAE of
0.05 means the prediction is off by 0.05 of the response scale. Squared error re-weights exactly the
sparse extremes that the density weighting is itself arguing about, so the metric and the
intervention would be entangled.

**Why the null is fitted out of fold.** The per-drug constant is fitted on each fold's *other* folds,
not on the rows it is scored against. A constant fitted on the held-out truth is an oracle: no real
predictor can achieve it, and comparing against it flatters the null and understates the model.

In [7]:
def values(y_true, y_pred):
    """Mean absolute error for one drug, in the target's own units.

    Absolute rather than squared so that neither loss arm is scored on its own objective (see above).
    Read directly: 0.05 means the prediction is off by five points of viability on average.

    :returns: the MAE, or ``nan`` on an empty input -- consistent with the other three quantities,
        where an undefined drug is excluded from the mean rather than contributing a value.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if y_true.shape != y_pred.shape:
        raise ValueError(f'shape mismatch: y_true {y_true.shape} vs y_pred {y_pred.shape}')
    if y_true.size == 0:
        return np.nan
    return float(np.mean(np.abs(y_true - y_pred)))


def oof_constant(frame):
    """The per-drug constant predictor, fitted OUT OF FOLD, aligned to ``frame``'s rows.

    For each (drug, fold), the mean of that drug's labels over every OTHER fold. Fitting on the same
    rows it is then compared against would let held-out labels define the baseline they are scored
    against -- the identical requirement ``dreval_normalize.naive_predictions`` enforces, and the
    reason the ``fold`` column is mandatory in §1.1.

    :returns: a Series aligned to ``frame.index``. ``nan`` where a drug has no rows outside its own
        fold, which cannot happen with >1 fold but is not silently filled if it does.
    """
    out = pd.Series(np.nan, index=frame.index, dtype=float)
    for (drug, fold), rows in frame.groupby(['drug', 'fold'], sort=False):
        others = frame[(frame['drug'] == drug) & (frame['fold'] != fold)]
        if len(others):
            out.loc[rows.index] = others['y_true'].mean()
    return out


## 1.8 · Scoring every arm

| | |
|---|---|
| **in** | both out-of-fold tables; the ridge baseline |
| **out** | `panel_metrics.csv`; the per-seed dispersion; the decision rule applied |

Applies §1.2–§1.7 to every arm of both architectures through the same code. Four choices this rests
on:

**Second baseline — the ridge control, not a per-cell-line mean over labels.** A per-line mean cannot
be fitted for a *held-out* line without that line's own labels, so under leave-cell-line-out it is
either an oracle or identically the global mean, since an unseen line's effect is zero. `RidgeCV` on
line-mean *embeddings* asks the same question legitimately, predicting a held-out line from its
expression.

**Dispersion across seeds, not folds.** Folds differ in which lines they hold out, so fold spread
confounds fold difficulty with model instability. Seeds isolate instability, which is what the column
is for.

**`SEED_BAND` is the half-range across the three seeds.** It is the observed spread of the runs
actually made, assumes no distribution, and is the more conservative of the two candidates; a
standard deviation over n = 3 is barely an estimate.

**The ridge control is reported beside the arms, not inside the table.** It is a different model
class — one prediction per cell line from a line-mean embedding — so folding it in invites reading it
as another arm of the same experiment.

In [8]:
# --- §1.4 · Score every arm, both architectures, through one scorer -------------------------
#
# THE DECISIONS THIS CELL RESTS ON (Selin, 13.08.2026), each recorded where it is used:
#
# 1. SECOND BASELINE -> the ridge control, NOT a per-cell-line mean over labels.
#    A per-line mean across drugs cannot be fitted for a HELD-OUT line without that line's own
#    labels, so under leave-cell-line-out it is either an oracle -- the "stricter variant" deleted
#    12.08.2026 as a local invention -- or identically the global mean, since an unseen line's
#    effect is zero. `RidgeCV` on line-mean EMBEDDINGS is the baseline that answers the same
#    question legitimately: it predicts a held-out line from its expression, not from its response.
# 2. `values` -> MEAN ABSOLUTE ERROR. Already what `values()` computes: it is in the target's own
#    units and directly readable, where squared error re-weights the tail the density sweep is
#    itself arguing about.
# 3. DISPERSION -> ACROSS SEEDS, not folds. Folds differ in which lines they hold out, so fold
#    spread confounds fold difficulty with model instability; seeds isolate instability, which is
#    what a dispersion column is for.
# 4. SEED_BAND -> HALF-RANGE across the three seeds. `decide()` refuses to guess, and says why: on
#    three seeds a half-range and a standard deviation differ by about a factor of two, and how the
#    recorded +-0.04 was computed is undocumented, so it cannot be inherited. The half-range is the
#    observed spread of the runs actually made, assumes no distribution, and is the wider (more
#    conservative) of the two. An sd over n=3 is barely an estimate.

SEED_BAND = lambda vals: (np.nanmax(vals) - np.nanmin(vals)) / 2.0

MIL_CSV = NB_DIR / 'outputs' / 'mil' / 'mil_oof_predictions.csv'
RIDGE_CSV = PANEL_OUT / 'panel_ridge_baseline.csv'

frames = [load_oof_table()]
if MIL_CSV.exists():
    frames.append(load_oof_table(MIL_CSV))
else:
    print(f'⚠️  {MIL_CSV.name} absent -- MIL is NOT scored. Run 4b, then re-run this cell.')
scored = pd.concat(frames, ignore_index=True)

QUANTITIES = {
    'order': order,
    'top_of_order': top_of_order,
    'values': values,
    'spread_slope': lambda t, p: calibration(t, p)[0],
    'spread_intercept': lambda t, p: calibration(t, p)[1],
}

rows = []
for arm, g in scored.groupby(ARM_KEYS, sort=False, dropna=False):
    rec = dict(zip(ARM_KEYS, arm))
    for name, fn in QUANTITIES.items():
        mean, used = across_drugs(per_drug(g, fn))
        rec[name] = mean
        rec[f'{name}_n_drugs'] = len(used)
    # the null every arm is read against: a per-drug constant fitted OUT OF FOLD
    rec['values_null'] = float(np.mean(np.abs(g['y_true'].to_numpy() - oof_constant(g).to_numpy())))
    rows.append(rec)

metrics = pd.DataFrame(rows)
metrics.to_csv(METRICS_CSV, index=False)
print(f'{len(metrics)} arms scored -> {METRICS_CSV.name}')
print()
print(metrics[ARM_KEYS + ['order', 'top_of_order', 'values', 'values_null',
                          'spread_slope', 'spread_intercept']].round(4).to_string(index=False))

# --- dispersion across seeds, and the decision rule -----------------------------------------
ACROSS = [k for k in ARM_KEYS if k != 'seed']
disp = (metrics.groupby(ACROSS, sort=False, dropna=False)[list(QUANTITIES)]
        .agg(['mean', 'std', 'count']).round(4))
print()
print('across seeds (mean / sd / n) -- sd is the dispersion, per decision 3 above:')
print(disp.to_string())

per_seed = {tuple(k): {q: g[q].tolist() for q in QUANTITIES}
            for k, g in metrics.groupby(ACROSS, sort=False, dropna=False)}
n_seeds = {k: len(v['order']) for k, v in per_seed.items()}
print()
if min(n_seeds.values(), default=0) < 3:
    print(f'⚠️  decide() needs >=3 seeds per arm; have {n_seeds}. Rule NOT applied.')
else:
    incumbent = next(k for k in per_seed if 'X_pca' in k and 0.0 in k)
    for challenger in per_seed:
        if challenger == incumbent:
            continue
        v = decide(per_seed, challenger, incumbent, seed_band=SEED_BAND)
        # `decide` returns 'challenger_wins' and 'per_quantity'; this read 'verdict', a key the
        # function has never returned, so §1.8 raised KeyError on its first execution (13.08.2026).
        # The failing quantities are named because the rule is "win on the primary, non-inferior on
        # every guard" -- which guard blocked it is the part worth reading.
        blocked = [q for q, r in v['per_quantity'].items() if not r['passed']]
        print(f'{challenger} vs {incumbent}: '
              f'{"WINS" if v["challenger_wins"] else "no"}'
              f'{"" if v["challenger_wins"] else "  (blocked on: " + ", ".join(blocked) + ")"}')

# The ridge control, reported beside the arms rather than folded into them -- it is a different
# model class, so it belongs next to the table, not in it.
if RIDGE_CSV.exists():
    ridge = pd.read_csv(RIDGE_CSV)
    print()
    print('ridge control (line-mean embeddings), mean Spearman per rep:')
    print(ridge.groupby('rep')['spearman'].mean().round(4).to_string())

42 arms scored -> panel_metrics.csv

    rep model  alpha loss  seed  order  top_of_order  values  values_null  spread_slope  spread_intercept
  X_pca   mlp    0.0  mse    42 0.2421        0.2136  0.0933       0.1002        0.6974            0.2531
  X_pca   mlp    0.0  mse    43 0.2365        0.2182  0.0932       0.1002        0.6406            0.3045
  X_pca   mlp    0.0  mse    44 0.2635        0.2318  0.0933       0.1002        0.7034            0.2389
  X_pca   mlp    0.0  mae    42 0.2615        0.2364  0.0927       0.1002        0.6891            0.2613
  X_pca   mlp    0.0  mae    43 0.2636        0.2227  0.0925       0.1002        0.6382            0.2989
  X_pca   mlp    0.0  mae    44 0.2601        0.2227  0.0929       0.1002        0.7027            0.2350
  X_pca   mlp    0.5  mse    42 0.2676        0.2909  0.0941       0.1002        0.6013            0.3263
  X_pca   mlp    0.5  mse    43 0.2815        0.2409  0.0942       0.1002        0.5770            0.3527
  X_pca  

---

# §2 · Absorb the diagnostics

| | |
|---|---|
| **in** | `outputs/diagnostics/*` — written by `analysis/evaluation/diagnostics.ipynb` |
| **out** | nothing; this section reads and reports |

**Why it is here.** `diagnostics` answers four questions the headline numbers cannot, plus a
dispersion read. Those answers belong in the evaluation chain rather than in a side notebook nobody
opens — but they are **not recomputed here**. Each quantity keeps its single owner: the artifact
`diagnostics` wrote. This section loads them and states what each says, so a reader of `5_evaluation`
meets them without having to know the other notebook exists.

**Nothing is selected.** All five CSVs are promoted, in the order `diagnostics` computes them. The
three PNGs are named rather than embedded, because a figure re-rendered here would be a second copy
of a drawing that already has an owner.

In [9]:
# --- §2 · Read the diagnostics artifacts and report what each establishes -----------------
#
# Absorbs analysis/evaluation/diagnostics.ipynb. Reads only -- every number below is owned by the
# CSV it is read from, never restated from memory, and this cell writes nothing.

DIAG = NB_DIR / 'outputs' / 'diagnostics'

missing = [f for f in ['gate_per_drug.csv', 'loss_share_per_line.csv', 'line_effect_vs_programs.csv',
                       'input_scale.csv', 'result_dispersion.csv'] if not (DIAG / f).exists()]
assert not missing, f'diagnostics has not been run, or wrote elsewhere: {missing}'

print('§2.1 · the drug-selection gate filtered on potency, not rankability')
gate = pd.read_csv(DIAG / 'gate_per_drug.csv').rename(columns={'Unnamed: 0': 'drug'})
print(f"  {len(gate)} drugs; auc_mean {gate.auc_mean.min():.3f}-{gate.auc_mean.max():.3f}, "
      f"auc_std {gate.auc_std.min():.3f}-{gate.auc_std.max():.3f}")
print(f"  correlation between potency (auc_mean) and spread (auc_std): "
      f"{spearmanr(gate.auc_mean, gate.auc_std).statistic:+.3f}   -> figure: gate_potency_vs_spread.png")

print('\n§2.2 · the per-cell loss weights cell lines by how many cells they contribute')
share = pd.read_csv(DIAG / 'loss_share_per_line.csv').rename(columns={'Unnamed: 0': 'cell_line'})
top = share.nlargest(1, 'loss_share').iloc[0]
print(f"  {len(share)} lines, {share.cells.min()}-{share.cells.max()} cells each "
      f"({share.cells.max() / share.cells.min():.0f}x)")
print(f"  largest single share {top.loss_share:.4f} ({top.cell_line}); "
      f"top 10 lines carry {share.nlargest(10, 'loss_share').loss_share.sum():.3f} of the loss")
print(f"  an equal split would give each line {1 / len(share):.4f}   -> figure: loss_share_per_line.png")

print('\n§2.3 · the cell-line effect is not proliferation')
prog = pd.read_csv(DIAG / 'line_effect_vs_programs.csv')
print(prog.reindex(prog.rho.abs().sort_values(ascending=False).index)
          .head(5).round(4).to_string(index=False))
print(f"  strongest |rho| over {len(prog)} programs: {prog.rho.abs().max():.3f}"
      f"   -> figure: line_effect_vs_proliferation.png")

print('\n§2.4 · the two representations enter the network on different scales')
scale = pd.read_csv(DIAG / 'input_scale.csv').set_index('rep')
ratio = scale.loc['X_pca', 'std_median'] / scale.loc['X_scGPT', 'std_median']
print(scale.round(4).to_string())
print(f"  median per-dimension sd ratio X_pca / X_scGPT = {ratio:.1f}x, "
      f"under ONE shared learning rate (TODO item 4A)")

print('\n§2.5 · how much do the results move?')
disp = pd.read_csv(DIAG / 'result_dispersion.csv')
print(disp.round(4).to_string(index=False))
print(f"  worst per-drug correlation anywhere: {disp.drug_min.min():+.3f}")

§2.1 · the drug-selection gate filtered on potency, not rankability
  534 drugs; auc_mean 0.333-1.065, auc_std 0.037-0.252
  correlation between potency (auc_mean) and spread (auc_std): -0.705   -> figure: gate_potency_vs_spread.png

§2.2 · the per-cell loss weights cell lines by how many cells they contribute
  181 lines, 56-1990 cells each (36x)
  largest single share 0.0434 (NCIH2110_LUNG); top 10 lines carry 0.182 of the loss
  an equal split would give each line 0.0055   -> figure: loss_share_per_line.png

§2.3 · the cell-line effect is not proliferation
        program     rho      p   n
ProtDegra_score -0.1902 0.0103 181
 ProtMatu_score -0.1607 0.0307 181
   p53Sen_score  0.0533 0.4758 181
     G1/S_score -0.0509 0.4963 181
   EpiSen_score  0.0509 0.4965 181
  strongest |rho| over 13 programs: 0.190   -> figure: line_effect_vs_proliferation.png

§2.4 · the two representations enter the network on different scales
         dims  std_median  std_max  std_min  within_rep_ratio  abs

---

# §3 · Absorb the external benchmark

| | |
|---|---|
| **in** | `outputs/dreval/*` — written by `analysis/evaluation/dreval_benchmark.ipynb` |
| **out** | nothing; this section reads and reports |

**Why it is here.** Everything else in this notebook scores the arms against each other under one
scorer of our own. `dreval_benchmark` scores them under **someone else's protocol**, against
baselines we did not choose, and it is the only external check the project has. It belongs beside
the internal result rather than in a separate notebook.

**Read the per-fold table, not the mean.** The benchmark's fold 1 has been unstable across
executions while folds 2–5 have not, so a single summary figure hides the one number that moves.
The tally below counts folds rather than averaging them, for that reason.

In [10]:
# --- §3 · Read the DrEval artifacts and report what they establish -------------------------
#
# Absorbs analysis/evaluation/dreval_benchmark.ipynb. Reads only; writes nothing.

DREVAL = NB_DIR / 'outputs' / 'dreval'
assert (DREVAL / 'dreval_lco_results.csv').exists(), 'dreval_benchmark has not been run'

lco = pd.read_csv(DREVAL / 'dreval_lco_results.csv')
print(f'§3.1 · DrEval leave-cell-line-out, {lco.fold.nunique()} folds, '
      f'{lco.algorithm.nunique()} algorithms')

onco = (lco[lco.algorithm.str.startswith('OncoMLP')]
        .pivot(index='fold', columns='algorithm', values='Spearman: normalized'))
onco.columns = [c.replace('OncoMLP ', '').strip('()') for c in onco.columns]
onco['ahead'] = np.where(onco.X_pca > onco.X_scGPT, 'PCA',
                         np.where(onco.X_scGPT > onco.X_pca, 'scGPT', 'tie'))
print(onco.round(4).to_string())
tally = onco.ahead.value_counts().to_dict()
print(f'  fold tally: {tally}')
print(f'  spread WITHIN X_pca across folds: {onco.X_pca.max() - onco.X_pca.min():.4f}')
print(f'  largest BETWEEN-arm gap in any fold: {(onco.X_pca - onco.X_scGPT).abs().max():.4f}')
print('  -> if the within-arm spread exceeds the between-arm gap, the protocol does not '
      'separate them')

print('\n§3.2 · against DrEval\'s own baselines, mean over folds')
base = (lco.groupby('algorithm')[['Spearman', 'Spearman: normalized']]
          .mean().sort_values('Spearman: normalized', ascending=False))
print(base.round(4).to_string())

print('\n§3.3 · DrEval normalization applied to the pipeline\'s own predictions')
norm = pd.read_csv(DREVAL / 'dreval_normalized.csv')
pooled = norm[norm.grouping == 'pooled']
print(pooled[['rep', 'alpha', 'n', 'Spearman', 'Spearman: normalized', 'R^2: normalized']]
      .round(4).to_string(index=False))
print('  -> figure: dreval_lco.png')

§3.1 · DrEval leave-cell-line-out, 5 folds, 10 algorithms
       X_pca  X_scGPT  ahead
fold                        
1     0.2331   0.2439  scGPT
2     0.2764   0.2903  scGPT
3     0.3576   0.2657    PCA
4     0.2625   0.2638  scGPT
5     0.2581   0.2962  scGPT
  fold tally: {'scGPT': 4, 'PCA': 1}
  spread WITHIN X_pca across folds: 0.1245
  largest BETWEEN-arm gap in any fold: 0.0918
  -> if the within-arm spread exceeds the between-arm gap, the protocol does not separate them

§3.2 · against DrEval's own baselines, mean over folds
                            Spearman  Spearman: normalized
algorithm                                                 
OncoMLP (X_pca)               0.7739                0.2776
SingleDrugRF (scgpt)          0.7756                0.2773
OncoMLP (X_scGPT)             0.7730                0.2720
SingleDrugEN (pca)            0.7699                0.2534
SingleDrugRF (pca)            0.7414                0.0279
NaiveCellLineMeanPredictor    0.0000             